In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

In [2]:
df = pd.read_csv('/datasets/users_behavior.csv')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [4]:
df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [5]:
df.describe()

,calls,minutes,messages,mb_used,is_ultra
count,3214.000000,3214.000000,3214.000000,3214.000000,3214.000000
mean,63.038892,438.208787,38.281269,17207.673836,0.306472
std,33.236368,234.569872,36.148326,7570.968246,0.461100
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,40.000000,274.575000,9.000000,12491.902500,0.000000
50%,62.000000,430.600000,30.000000,16943.235000,0.000000
75%,82.000000,571.927500,57.000000,21424.700000,1.000000
max,244.000000,1632.060000,224.000000,49745.730000,1.000000


In [6]:
features = df.drop('is_ultra', axis=1)
target = df['is_ultra']
features_train, features_test, target_train, target_test = train_test_split(
    features, target, 
    test_size=0.4, 
    random_state=54321
)
features_test, features_valid, target_test, target_valid  = train_test_split(
    features_test,
    target_test,
    test_size=0.5,
    random_state=1189
)

print(features_train.shape)
print(features_valid.shape)
print(features_test.shape)

(1928, 4)
(643, 4)
(643, 4)


Calculé y probé los 3 modelos localmente (Arbol de decisión, Bosque aleatorio y Regresión logística)
y comprobé que para esto el que mejores resultados da es el Bosque aleatorio, así que ahora comprobaré cuales son los mejores hiperparametros

In [7]:
#PRUEBA DE HIPERPARAMETROS USANDO EL MODELO RandomForestClassifier
best_accuracy = 0
best_est = 0
best_depth = 0

for est in range(10, 51 , 10):
    for depth in range(1, 31):  
      
        model = RandomForestClassifier(
            random_state=54321,
            n_estimators=est,
            max_depth=depth
        )
        
        model.fit(features_train, target_train)
        predictions_valid = model.predict(features_valid)
        accuracy = accuracy_score(target_valid, predictions_valid)
        print(
            "Exactitud para n_estimators =", est,
            ", max_depth =", depth,
            "es", accuracy
        )
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_est = est
            best_depth = depth
print(
    "Exactitud del mejor modelo en el conjunto de validación:",
    best_accuracy,
    "n_estimators:",
    best_est,
    "max_depth:",
    best_depth

)

#Solo quería probar diferentes n_estimators y max_depth para ver si era posible llegar a mas de 81.49% de exactitud, pero creo que los datos no lo permiten

Exactitud para n_estimators = 10 , max_depth = 1 es 0.7589424572317263
Exactitud para n_estimators = 10 , max_depth = 2 es 0.7791601866251944
Exactitud para n_estimators = 10 , max_depth = 3 es 0.7807153965785381
Exactitud para n_estimators = 10 , max_depth = 4 es 0.7962674961119751
Exactitud para n_estimators = 10 , max_depth = 5 es 0.7978227060653188
Exactitud para n_estimators = 10 , max_depth = 6 es 0.7978227060653188
Exactitud para n_estimators = 10 , max_depth = 7 es 0.80248833592535
Exactitud para n_estimators = 10 , max_depth = 8 es 0.8087091757387247
Exactitud para n_estimators = 10 , max_depth = 9 es 0.7993779160186625
Exactitud para n_estimators = 10 , max_depth = 10 es 0.8149300155520995
Exactitud para n_estimators = 10 , max_depth = 11 es 0.8040435458786936
Exactitud para n_estimators = 10 , max_depth = 12 es 0.7916018662519441
Exactitud para n_estimators = 10 , max_depth = 13 es 0.7947122861586314
Exactitud para n_estimators = 10 , max_depth = 14 es 0.7900466562986003
Exa

Conclusión:
El mejor modelo fue un Random Forest con n_estimators=10 y max_depth=10, alcanzando una exactitud de 81.49% en el conjunto de validación. Este resultado supera el objetivo establecido de 75%, por lo que el modelo cumple con los requisitos del proyecto.

In [8]:
model = RandomForestClassifier(
            random_state=54321,
            n_estimators=10,
            max_depth=10
        )
model.fit(features_train, target_train)
prediction_test = model.predict(features_test)
accuracy = accuracy_score(target_test, prediction_test)
print(accuracy)

0.8118195956454122


In [9]:
target_train.value_counts(normalize=True)

0    0.697095
1    0.302905
Name: is_ultra, dtype: float64

Acá hago la prueba de cordura, la proporción real de los clientes es aproximadamente 70/30, un modelo "tonto" que se inclina solo a la clase
mayoritaria acertaría 7 de cada 10 veces, es decir, tiene una exactitud del 70%, mi modelo tiene 81%, quiere decir que aprende patrones reales
y pasa la prueba de cordura